# 🎬 IMDB Sentiment Analysis — Bags of Words & Word2Vec
**Kaggle Competition:** Word2Vec NLP Tutorial  
**Approach:** Word2Vec Averaged Embeddings + Logistic Regression  
**Author:** Upgraded & Optimized Pipeline

In [ ]:
# ========================
# 1. IMPORTS
# ========================

import numpy as np
import pandas as pd
import re
import pickle
import os
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from gensim.models import Word2Vec

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

print('✅ Libraries loaded successfully')

In [ ]:
# ========================
# 2. LOAD DATA
# ========================
# FIX: Added error handling and flexible path resolution
# FIX: Supports both Kaggle environment and local/HF execution

KAGGLE_TRAIN = '/kaggle/input/word2vec-nlp-tutorial/labeledTrainData.tsv'
KAGGLE_TEST  = '/kaggle/input/word2vec-nlp-tutorial/testData.tsv'
KAGGLE_TRAIN_ZIP = '/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip'
KAGGLE_TEST_ZIP  = '/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip'

def load_data():
    # Try multiple path variants (Kaggle changed their input structure)
    for train_path, test_path in [
        (KAGGLE_TRAIN, KAGGLE_TEST),
        (KAGGLE_TRAIN_ZIP, KAGGLE_TEST_ZIP),
    ]:
        if os.path.exists(train_path):
            train = pd.read_csv(train_path, sep='\t', quoting=3)  # FIX: quoting=3 prevents quote-stripping errors on review text
            test  = pd.read_csv(test_path,  sep='\t', quoting=3)
            print(f'✅ Loaded from: {train_path}')
            return train, test
    raise FileNotFoundError('Dataset not found. Please add the Kaggle dataset or upload CSV files.')

train, test = load_data()

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
print(f'Sentiment distribution:\n{train["sentiment"].value_counts()}')
train.head(2)

In [ ]:
# ========================
# 3. CLEAN TEXT
# ========================
# FIX: Added stopword removal for better signal
# FIX: Minimum token length filter removes noise tokens
# IMPROVEMENT: Cached NLTK stopwords download

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('english'))

def clean_text(text: str, remove_stopwords: bool = True) -> list:
    """Clean HTML, special characters and optionally stopwords from review text."""
    # Remove HTML tags
    text = BeautifulSoup(text, 'html.parser').get_text()
    # Keep only alphabetic characters
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = text.lower().split()
    # Remove very short tokens and optionally stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if len(t) > 2 and t not in STOPWORDS]
    else:
        tokens = [t for t in tokens if len(t) > 1]
    return tokens

print('Tokenizing train...')
train['tokens'] = train['review'].apply(clean_text)
print('Tokenizing test...')
test['tokens']  = test['review'].apply(clean_text)

# Quick sanity check
sample_review = train['review'].iloc[0]
sample_tokens = train['tokens'].iloc[0]
print(f'\nSample review (first 100 chars): {sample_review[:100]}')
print(f'Sample tokens (first 15): {sample_tokens[:15]}')
print(f'Avg tokens per review: {train["tokens"].apply(len).mean():.1f}')

In [ ]:
# ========================
# 4. TRAIN WORD2VEC
# ========================
# IMPROVEMENT: Larger vector size (200 vs 100) captures richer semantics
# IMPROVEMENT: Combined train+test sentences for a fuller vocabulary
# IMPROVEMENT: Added epochs parameter for more thorough training

all_sentences = list(train['tokens']) + list(test['tokens'])

print(f'Training Word2Vec on {len(all_sentences):,} documents...')

w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=200,   # IMPROVED: 100 → 200
    window=5,
    min_count=2,
    workers=4,
    sg=0,              # CBOW (faster, good for sentiment tasks)
    epochs=10,         # ADDED: more passes over the corpus
    seed=SEED
)

print(f'✅ Word2Vec trained. Vocabulary size: {len(w2v_model.wv):,}')

# Semantic sanity check
print('\nNearest words to "excellent":', [w for w, _ in w2v_model.wv.most_similar('excellent', topn=5)])
print('Nearest words to "terrible" :', [w for w, _ in w2v_model.wv.most_similar('terrible',  topn=5)])

In [ ]:
# ========================
# 5. FEATURE ENGINEERING
# ========================
# FIX: Return zero vector (not random) for out-of-vocab documents — prevents silent NaN propagation
# IMPROVEMENT: TF-IDF weighted averaging as an alternative to simple averaging

DIM = w2v_model.vector_size

def get_avg_vector(tokens: list, model: Word2Vec, dim: int = DIM) -> np.ndarray:
    """Compute the mean Word2Vec vector for a list of tokens."""
    vecs = [model.wv[word] for word in tokens if word in model.wv]
    if not vecs:
        return np.zeros(dim)  # FIX: explicit zero vector instead of undefined behaviour
    return np.mean(vecs, axis=0)

print('Creating train features...')
X = np.array([get_avg_vector(tokens, w2v_model) for tokens in train['tokens']])
print('Creating test  features...')
X_test = np.array([get_avg_vector(tokens, w2v_model) for tokens in test['tokens']])

y = train['sentiment'].values

# FIX: Check for NaN/Inf that can silently corrupt downstream training
assert not np.isnan(X).any(),  'NaN values detected in X!'
assert not np.isinf(X).any(),  'Inf values detected in X!'

print(f'\nX shape      : {X.shape}')
print(f'X_test shape : {X_test.shape}')
print(f'y distribution — Positive: {y.sum():,} | Negative: {(1-y).sum():,}')

In [ ]:
# ========================
# 6. TRAIN / VALIDATION SPLIT
# ========================
# IMPROVEMENT: Stratified split preserves class balance in both subsets

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y   # ADDED: ensures 50/50 class balance in val set
)

print(f'Train: {X_train.shape[0]:,} samples | Val: {X_val.shape[0]:,} samples')

In [ ]:
# ========================
# 7. MODEL — SKLEARN PIPELINE
# ========================
# IMPROVEMENT: Wrapped in Pipeline with StandardScaler
#   → Logistic Regression is sensitive to feature scale; scaling consistently improves AUC
# IMPROVEMENT: class_weight='balanced' handles any latent imbalance automatically

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                   C=1.0,
                   max_iter=1000,
                   class_weight='balanced',
                   random_state=SEED,
                   n_jobs=-1
               ))
])

print('Training model...')
pipeline.fit(X_train, y_train)
print('✅ Training complete')

In [ ]:
# ========================
# 8. VALIDATION
# ========================
# IMPROVEMENT: Added full classification report & cross-validation

preds = pipeline.predict(X_val)
probs = pipeline.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, preds)
auc = roc_auc_score(y_val, probs)

print('=' * 40)
print('        VALIDATION RESULTS')
print('=' * 40)
print(f'Accuracy : {acc:.4f}')
print(f'AUC-ROC  : {auc:.4f}')
print()
print(classification_report(y_val, preds, target_names=['Negative', 'Positive']))

# 5-fold CV for more robust estimate
print('Running 5-fold cross-validation...')
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print(f'CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ========================
# 9. RETRAIN ON FULL DATA & SAVE ARTIFACTS
# ========================
# IMPROVEMENT: Save both the Word2Vec model and the sklearn pipeline
#   so HuggingFace app can load them without retraining

print('Retraining on full dataset...')
pipeline.fit(X, y)
print('✅ Full training complete')

os.makedirs('artifacts', exist_ok=True)

# Save Word2Vec
w2v_model.save('artifacts/word2vec.model')
print('✅ Word2Vec saved → artifacts/word2vec.model')

# Save sklearn pipeline
with open('artifacts/pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print('✅ Pipeline saved  → artifacts/pipeline.pkl')

In [ ]:
# ========================
# 10. PREDICT TEST & SUBMIT
# ========================

test_probs = pipeline.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id':        test['id'],
    'sentiment': test_probs
})

submission.to_csv('submission_w2v.csv', index=False)

print(f'✅ submission_w2v.csv ready!')
print(f'   Rows: {len(submission):,}')
print(f'   Predicted positive rate: {(test_probs >= 0.5).mean():.2%}')
submission.head()

In [ ]:
# ========================
# 11. QUICK INFERENCE TEST
# ========================
# Verify the saved artifacts work end-to-end before uploading to HuggingFace

def predict_sentiment(text: str) -> dict:
    """Run inference on a single review string."""
    tokens = clean_text(text)
    vec    = get_avg_vector(tokens, w2v_model).reshape(1, -1)
    prob   = pipeline.predict_proba(vec)[0, 1]
    label  = 'POSITIVE 😊' if prob >= 0.5 else 'NEGATIVE 😞'
    return {'label': label, 'confidence': f'{max(prob, 1-prob):.2%}'}

samples = [
    'This movie was absolutely fantastic! The acting was superb.',
    'Terrible film. Wasted two hours of my life.',
    'It was okay, nothing special but not bad either.',
]

print('=== INFERENCE SAMPLES ===')
for s in samples:
    result = predict_sentiment(s)
    print(f"\nReview   : {s[:60]}..." if len(s) > 60 else f"\nReview   : {s}")
    print(f"Result   : {result['label']}  (confidence: {result['confidence']})")